<a href="https://colab.research.google.com/github/jiraroj-wir/MUIC-ICCS261-Principles-of-Data-Science/blob/main/Copy_of_05_DataManipulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Manipulation

*Originally from <code>pandas</code>'s [Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html)<br>
Modified by B. Mathayomchan and K. Bunchongchit*<br>
Last edited on May 6, 2026

## Introduction

Data that are perfectly structured in one file ready for your data science tasks are extremely rare. In real life, we need to construct data from different sources and files before we can start to explore and understand how these data could be useful to our research question. This notebook lists basic commands in pandas that help us shape our data.

## Basic Data Manipulation Methods
- Combining Data Sets
    - Adding rows from a dataframe with identical columns
    - Adding a dataframe where columns are not identical
    - Adding Columns
    - Adding along the direction of column
- Merging (combining dataframes base on values)
  - One-to-One Merge
  - Many-to-One Merge
  - One-to-Many Merge
  - Many-to-Many Merge
- Joining on keys

### Combining Data Sets

Three common methods:

- `pandas.concat()`: across rows or columns
- `DataFrame.merge()`: on common columns or indices
- `DataFrame.join()`: on a key column or an index

*Reference*
- [Three ways to combine DataFrames in Pandas](https://www.educative.io/answers/three-ways-to-combine-dataframes-in-pandas)

### Adding rows from a dataframe with identical columns

In [ ]:
import pandas as pd
import numpy as np

CSV_URL = 'https://drive.google.com/uc?export=download&id='

The following code is for **Google Colab only**.

In [ ]:
# Load muic1.csv
df1 = pd.read_csv(CSV_URL + '1W_rGyWriKyYk0yFkwwO_3KNmJWPxwsn1')
# Load muic2.csv
df2 = pd.read_csv(CSV_URL + '1QCbItzfbLDgojZkZwWU6sMi2HZ4c03oD')
# Load muic3.csv
df3 = pd.read_csv(CSV_URL + '1-yHsuPzNn7sBj4FWT58YPcQGZ1yD5no1')

If you use a **desktop IDE**,
1. Download the data files from https://drive.google.com/drive/folders/1all1VgS-0bFr9uB27NWG8UXzrIBTh2Xl?usp=sharing
2. Replace both lines below with your local file paths, such as
```python
df1 = pd.read_csv('./datasets/muic1.csv')
df2 = pd.read_csv('./datasets/muic2.csv')
df3 = pd.read_csv('./datasets/muic3.csv')
```

In [ ]:
df1

In [ ]:
df2

In [ ]:
df3

In [ ]:
row_concat1 = pd.concat([df1, df2, df3])
row_concat1.shape

In [ ]:
row_concat1

Notice the order of cancatenation as indicated in the image below.

<img src="https://drive.google.com/uc?export=view&id=1xy_hOqFhjBUckEqN3wP9qdCX_Fyz-fYn" width="40%">

Subset the 4<sup>th</sup> student out, notice that there are many rows in index 3 since they all match the label.

In [ ]:
row_concat1.loc[3]

To get the only the 4th row, we use `iloc`. `iloc` is the integer location, so only one row will be returned.

In [ ]:
row_concat1.iloc[3]

Doing it this way will have a confusing index. So, we may want to ignore the original index.

In [ ]:
row_concat2 = pd.concat([df1, df2, df3], ignore_index=True)
row_concat2  # notice the nicely running index below

### Adding a dataframe where columns are not identical
#### Case I : A subset

Recall `df1`.

In [ ]:
df1

In [ ]:
# Create a dataframe with fewer columns (to be used next)
narrower_df = pd.DataFrame({
    "ID": [6781111, 6781112],
    "Name": ["Jacky Chow", "Jet Loo"]
    })
narrower_df

```df1``` has columns: `ID`, `Name`, `Major`, `GPA`<br>
```narrower_df``` has columns: `ID`, `Name`<br>
Data will be concatenated nicely, and the one that doesn't have data will be ```NaN```.

In [ ]:
dfx = pd.concat([df1, narrower_df])
dfx

The image below summarizes the operation above.<br>
<img src="https://drive.google.com/uc?export=view&id=1wILEZi4X7mwc3j0VV2dV99GQOREHU4Pl" width="50%">

#### Case II : totally different

In [ ]:
# Create a dataframe with totally different columns (to be used next)
another_df = pd.DataFrame({
    "Hobby": ["Reading", "Running"],
    "Club": ["Board Game", "Debate"]
    })
another_df

```df1``` &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; has columns: ```ID, Name Major, GPA```<br>
```another_df``` has  columns: ```..................., Hobby, Club```

There will be all columns from both dataframes. For each row, any missing values from newly added columns will be NaN (aka missing).

In [ ]:
dfy = pd.concat([df1, another_df])
dfy

#### Case III : some columns are the same

In [ ]:
# Create a dataframe with some columns are the same (to be used next)
foo_df = pd.DataFrame({
    "ID":[6780101, 6780102],
    "Major": ["ICCH", "ICCS"],
    "Weight": [65, 59],
    "Height": [170, 180]})
foo_df

```df1``` has&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;columns: ```ID, Name, Major, GPA```<br>
```foo_df``` has  columns: ```ID, ..... Major, .... Weight, Height```<br>
Notice the results, those same columns will have data from both Dataframe objects.

In [ ]:
dfz = pd.concat([df1, foo_df])
dfz

###`inner`

What if we only care for columns that appear on both dataframes? Only columns with values appear on both dfs will be here.


```df1``` has&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;columns: ```ID, Name, Major, GPA```

```foo_df``` has  columns: ```ID, ..... Major, .... Weight, Height```


Notice only ID,Major are created.

In [ ]:
dfz = pd.concat([df1, foo_df], join="inner")
dfz

- The `inner` keyword will look at the structures of both dataframes. If any columns are available on both, those columns will be in the result.
- Hence, this is **structural**.
- It doesn't look at the value inside the cells. Let's try to test it. We will set one cell as NaN (missing). We will see what happens.


In [ ]:
# Let's force 6780001 (Ann Molive) to have NaN in major field
df1.loc[0, 'Major'] = np.nan
df1

In [ ]:
foo_df

```df1``` has&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;columns: ```ID, Name, Major, GPA```<br>
```foo_df``` has  columns: ```ID, ..... Major, .... Weight, Height```<br>


Notice that the result have NaN in the cell. ```concat``` doesn't concatenate based on value, but on structures.

In [ ]:
dfz = pd.concat([df1, foo_df], join="inner")
dfz

### outer
is the default value of the ```concat``` function.

So, we don't need to specify it. Leaving it blank is the same as setting join="outer"

```df1``` has&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;columns: ```ID, Name, Major, GPA```<br>
```foo_df``` has  columns: ```ID, ..... Major, .... Weight, Height```

In [ ]:
dfz = pd.concat([df1, foo_df], join="outer")
dfz

### Adding Columns

The following code is for **Google Colab** only.

In [ ]:
CSV_URL = 'https://drive.google.com/uc?export=download&id='
# Load concat_1.csv
df1 = pd.read_csv(CSV_URL + '1V6MG7Wecmvh-tgwojTqwu0HUVYhihHQ9')
# Load concat_2.csv
df2 = pd.read_csv(CSV_URL + '12ZEScK0XGAVR0TXwsKLVoI3HsAL1c9ko')
# Load concat_3.csv
df3 = pd.read_csv(CSV_URL + '1bJV3IleIN-GC-k-I07eKfIAa_dmanKCD')

In [ ]:
df1

In [ ]:
df2

In [ ]:
df3

Calling concat will just append rows below as we have seen earlier.

In [ ]:
row_concat = pd.concat([df1, df2, df3])
row_concat

### What if we want to add along the direction of column?
#### Adding all columns by concat with axis = 0
axis 0 is "row".
data with the same row labels will be in the same row

In [ ]:
col_concat = pd.concat([df1, df2, df3], axis=1)
col_concat

In [ ]:
col_concat['A']

#### Adding columns with different indexes

First, let's rename the column names

In [ ]:
df1.columns = ['A', 'B', 'C', 'D']
df2.columns = ['D', 'E', 'F', 'G']
df3.columns = ['A', 'D', 'G', 'H']

In [ ]:
df1

In [ ]:
df2

In [ ]:
df3

In [ ]:
# df1 has columns: A,B,C,D
# df2 has columns:       D,E,F,G
# df3 has columns: A,    D,    G,H
# notice when columns appear on multiple dataframes
row_concat = pd.concat([df1, df2, df3])
row_concat

In [ ]:
# df1 has columns: A,B,C,D
# df2 has columns:       D,E,F,G
# df3 has columns: A,    D,    G,H
# Notice that all dataframes have column D
row_concat = pd.concat([df1, df2, df3], join='inner')
row_concat

#### In what situation we can use this method?
Sometimes, we have dataframs that carry differnt number of columns. If both of them carry the same important columns, we may want to do the "inner" join so eliminate extra columns from each dataframe.

In [ ]:
# Create a dataframe to be used next
addr1_df = pd.DataFrame({
    "Number":["114", "153/1", "102", "945"],
    "Building":["A", "Beach Front", "Main bldg", "C"],
    "Street":["Sai 4", "Rama IX", "Sai 3", "Baromratchonnee"],
    "City":["Salaya", "Payathai", "Salaya", "Aroonamarin"],
    "District":["Taweewattana", "Ratchtevee", "Taweewattana", "Bangkok Noi"],
    "Province":["Bangkok", "Bangkok", "Bangkok", "Bangkok"],
    "ZIP":[73710, 10400, 73710, 10700]})

addr1_df

In [ ]:
# Create a dataframe to be used next
addr2_df = pd.DataFrame({
    "Number":["208", "2491", "327", "123"],
    "Street":["Rama VI", "Rama I", "Sukhumvit", "Charansanitwong"],
    "City":["SamSen Nai", "Bangrug", "Prakanong", "Bangplud"],
    "District":["Phayathai", "Ratchtevee", "Taweewattana", "Bangkok Noi"],
    "Province":["Bangkok", "Bangkok", "Bangkok", "Bangkok"],
    "Country":["Thailand", "Thailand", "Thailand", "Thailand"],
    "ZIP":[10100, 10400, 10600, 10200]
})
addr2_df

In [ ]:
# If we concat with default join ("outer"), we will get many fields with NaN
addr_df = pd.concat([addr1_df, addr2_df])
addr_df

In [ ]:
# This is more concise
addr_df = pd.concat([addr1_df, addr2_df], join="inner")
addr_df

#### Concatenate Columns With Different Rows

In [ ]:
# Change the row labels.
df1.index = [0, 1, 2, 3]
df2.index = [2, 3, 4, 5]
df3.index = [2, 3, 6, 7]

In [ ]:
df1

In [ ]:
df2

In [ ]:
df3

In [ ]:
# df1 has rows: 0,1,2,3
# df2 has rows:     2,3,4,5
# df3 has rows:     2,3,   6,7
col_concat = pd.concat([df1, df2, df3], axis=1)
col_concat

Notice that the row labels indicate what data will be in that row. At the end, we have row 0, 1, 2, 3, 4, 5, 6, and 7.

Also, notice the column labels, many of them have the same name !!!

In [ ]:
# select columns with label "A" out
col_concat["A"]

In [ ]:
# set join = "inner"
# if we want only rows that appear in all dataframes
col_concat = pd.concat([df1, df2, df3], axis=1, join="inner")
col_concat

In [ ]:
pd.concat([df2, df1, df3], axis=1)

In [ ]:
pd.concat([df1, df2, df3], axis=0)

### Recap on `concat`

- Concatenate `DataDrame` objects
- Three commonly-used paprameters:
    - **`axis`**
        - 0 by default, row-wise cacatneation
        - 1 = column-wise cacatneation
    - **`ignore_index`**
        - If`True`, source objects' indices are ignore, and 0, 1, 2, ... will be used instead.
    - **`join`**
        - **`outer`**: the default axis
        - **`inner`**: keeps only rows and/or columns with shared labels/indices

## Merging - Combining dataframes based on values
There will be cases that we need to merge data in a more specific way. Pandas provide many functions to do this.

We will use data from the reference textbook of this course, "Pandas for Everyone".<br>
Alternative resource: [Combining Datasets: Merge and Join](https://jakevdp.github.io/PythonDataScienceHandbook/03.07-merge-and-join.html)

The following code is for Google Colab only. If you use a desktop IDE,
1. Download the data files from https://drive.google.com/drive/folders/1all1VgS-0bFr9uB27NWG8UXzrIBTh2Xl?usp=sharing
2. Replace both lines below with your local file paths, such as
```python
person = pd.read_csv("./datasets/survey_person.csv")
site = pd.read_csv("./datasets/survey_site.csv")
survey = pd.read_csv("./datasets/survey_survey.csv")
visited = pd.read_csv("./datasets/survey_visited.csv")
```

In [ ]:
# Load the file survey_person.csv
person = pd.read_csv(CSV_URL + '1OMsps89VPOY_egCvhI9gLLULngGqDdBH')
# Load the file survey_site.csv
site = pd.read_csv(CSV_URL + '1isk_-IhK0BWkr2pTpNSQkID6JoiZmrDO')
# Load the file survey_survey.csv
survey = pd.read_csv(CSV_URL + '1gHjkhtDOzdd-D_IK-W12L3pQOH_nwqcn')
# Load the file survey_visited.csv
visited = pd.read_csv(CSV_URL + '1XX_FdD36zk7IgFeeOBEY5wfsAn4TCtwd')

In [ ]:
person

In [ ]:
site

In [ ]:
survey

In [ ]:
visited

### One-to-One Merge
The column we wants to join has no duplicated value.
Each value in the join-column of the left dataframe occurs once.
Each value in the join-column of the right dataframe occurs once.

Notice each value of the **name** column of the **site** dataframe occurs only once.

Notice each value of the **site** column of the **visited_subset** dataframe occurs only once.

In [ ]:
# Let's select only some rows, so it is easier to observe changes
visited_subset = visited.loc[[0, 2, 6], ]
visited_subset

In [ ]:
# Notice the site column here
site

In [ ]:
# LEFT  dataframe: site           : has columns:       name,lat,long
# RIGHT dataframe: visited_subset : has columns: ident,site,         dated
#
# noted: both dataframes have different names to store the site name.
# In terms of value: LEFT->name is the same as RIGHT->site
one2one_merge = site.merge(visited_subset, left_on="name", right_on="site")
one2one_merge

In [ ]:
one2one_join = site.join(visited_subset)
one2one_join

### Many-to-One Merge
One of the dataframes has repeat values.

Notice (in the cell below) some values of the **site** column of the **visited** dataframe occurs more than once (such as DR-3).

Notice each value of the **name** column of the **site** dataframe occurs only once (such as DR-3 here appears only once).



In [ ]:
# Let's modify visited a bit, we will see the effect that next
visited.loc[2, 'site'] = np.nan
visited

*References*
- [Merge Two pandas DataFrames in Python (6 Examples)(https://statisticsglobe.com/merge-two-pandas-dataframes-python)
- https://deepage.net/features/pandas-merge.html

| Join | Description |
|---|---|
| `Left` | _Keep all the keys from the left_ |
| `Right` | _Keep all the keys from the right_ |
| `outer` | _Keep all the keys from both left and right_ |
| `inner` | _Keep only keys that exist in both left and right_ |

In [ ]:
site

In [ ]:
# LEFT: visited has columns: ident,site,dated
# RIGHT: site has   columns:                  name,lat, long
Many2One_merge = visited.merge(site, left_on="site", right_on='name')
Many2One_merge

Notice the result will have columns: ident,site,dated,name,lat,long
For each row in LEFT, it'll look at the value in column "site" of that row.
With this value, it will look in to the table from the RIGHT.
If there is a match, the row from the right will be added (=more columns)
If there is no match, it won't show up here.

### One-to-Many Merge
The concept is the same, we just swap the order

In [ ]:
# LEFT: site has columns: name,lat, long
# RIGHT: visited has columns: ident,site,dated
One2Many_merge = site.merge(visited, left_on="name", right_on='site')
One2Many_merge

For each row in LEFT, it'll look at the value in column "name" of that row.
With this value, it will look in to the table from the RIGHT.
- If there is a match, the row from the right will be added (= more columns)
- If there is no match, it won't show up here.
- Where there are multiple matches, the right appear in more than one row.
- For each successful match, it will become one row.

### Many-to-Many Merge
The data on both sides contain duplicated values.

Let's start with a Many-to-One

In [ ]:
# Create a dataframe to be used next
# Notice that the many students take the same course

enrollment_df = pd.DataFrame({
    "ID": [1001,1002],
    "Course": ["ICCS161", "ICCS161"]
})
enrollment_df

In [ ]:
# Create another dataframe to be used next
# Notice that one course has many exams

exam_df = pd.DataFrame({
    "Course": ["ICCS161", "ICCS161"],
    "Exam": ["Midterm", "Final"]
})
exam_df

In [ ]:
# MANY-TO-MANY
# LEFT: enrollment_df
# RIGHT: exam_df
Many2Many_merge = enrollment_df.merge(exam_df, left_on="Course",right_on="Course")
Many2Many_merge

For each row in LEFT, it'll look at the value in column "Course" of that row.
With this value, it will look in to the table from the RIGHT.
- If there is a match, the row from the right will be added (=more columns)
- If there is no match, it won't show up here.
- Where, there are multiple matches. Since the right appear in more than one row.
- For each successful match, it will become one row.

This is called Many-to-Many because MANY students enroll the same course.
One Course has many exams (midterm & final). Therefore, when we think about  STUDETS---EXAM. That's many to many.

Let's consider another example.

In [ ]:
enrollment_df_2 = pd.DataFrame({
    "ID": [1001, 1002, 1003],
    "Course": ["ICCS161", "ICCS161", "ICCS331"]
})
enrollment_df_2

In [ ]:
Many2Many_merge_2 = enrollment_df_2.merge(exam_df, left_on="Course", right_on="Course")
Many2Many_merge_2

### `merge`'s Parameters

![`inner` vs `outer`](https://i.sstatic.net/1rb1R.jpg)<br>Ref: [pandas Merging 101](https://stackoverflow.com/questions/53645882/pandas-merging-101)

###`join`

So far we have worked with dataframes with no row labels. Let's add them and see the effect of the index label. Now we can perform <code>join()</code> that combines columns of two dataframes with different indices.

<table>
    <tr>
        <td><img src="https://i.sstatic.net/3qpXx.gif"></td>
        <td><img src="https://i.sstatic.net/dG8mw.gif"></td>
    </tr>
    <tr>
        <td><img src="https://i.sstatic.net/JpPRH.gif"></td>
        <td><img src="https://i.sstatic.net/s5hgJ.gif"></td>
    </tr>
</table>

Ref: [pandas Merging 101](https://stackoverflow.com/questions/53645882/pandas-merging-101)

In [ ]:
enrollment_df_2.set_index('Course', inplace=True)
enrollment_df_2

In [ ]:
exam_df.set_index('Course', inplace=True)
exam_df

In [ ]:
Many2Many_merge_3 = enrollment_df_2.join(exam_df, how="outer")
Many2Many_merge_3

In [ ]:
Many2Many_merge_3 = enrollment_df_2.join(exam_df, how="inner")
Many2Many_merge_3

### Conclusions

#### Similarities and Differences of `concat`, `merge`, `join`

Find the operation equivalent to the LHS.

In [ ]:
import pandas as pd

CSV_URL = 'https://drive.google.com/uc?export=download&id='
# Load concat_1.csv
df1 = pd.read_csv(CSV_URL + '1V6MG7Wecmvh-tgwojTqwu0HUVYhihHQ9')
# Load concat_2.csv
df2 = pd.read_csv(CSV_URL + '12ZEScK0XGAVR0TXwsKLVoI3HsAL1c9ko')

In [ ]:
df3 = df2.copy()

df3.iloc[2] = df1.iloc[1]
df3.iloc[3] = df1.iloc[2]

df2.columns = ['D', 'E', 'F', 'G']
df3.columns = ['A', 'D', 'G', 'H']

df2.index=[0, 2, 3, 5]
df3.index=[2, 3, 6, 7]

In [ ]:
df1

In [ ]:
df2

In [ ]:
df3

In [ ]:
pd.concat([df1, df2])

In [ ]:
pd.concat([df1, df2], axis=1)

In [ ]:
pd.concat([df1, df2], join='inner')

In [ ]:
pd.concat([df1, df2], axis=1, join='inner')

In [ ]:
df1.join(df2, lsuffix="_left", rsuffix="_right", how='outer')

In [ ]:
pd.merge(df1, df2, left_index=True, right_index=True)

In [ ]:
df1.join(df2, lsuffix="_left", rsuffix="_right", how='left')

In [ ]:
pd.merge(df1, df2, left_index=True, right_index=True, how='left')